## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Pengujian Asumsi Statistik](images/img_04_statistical_assumptions.png)

```
                   +-----------------------------+
                   |     DATASET NUMERIK         |
                   +-----------------------------+
                                  |
            +---------------------+---------------------+
            |                                           |
            v                                           v
  [1] UJI NORMALITAS                         [2] UJI HOMOGENITAS
  Shapiro-Wilk / Q-Q Plot                    Levene's Test
  p-val > 0.05 -> Normal                     p-val > 0.05 -> Homogen
  p-val <= 0.05 -> Non-Normal                p-val <= 0.05 -> Heterogen
            |                                           |
            +---------------------+---------------------+
                                  |
                                  v
                    [3] UJI MULTIKOLINEARITAS
                    VIF (Variance Inflation Factor)
                    VIF < 5 -> Tidak Ada Multikolinearitas
                    VIF >= 10 -> Terjadi Multikolinearitas Parah
```


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)

# Memuat dataset server performance
df_server = pd.read_csv("../datasets/02_server_performance_assumptions.csv")
print("Data kinerja server berhasil dimuat. Total:", len(df_server))
display(df_server.head())


## 🧪 3. Pengujian Normalitas Data (Shapiro-Wilk & Visual Q-Q Plot)


In [ ]:
stat_shapiro, p_shapiro = stats.shapiro(df_server['response_latency_ms'])
stat_ks, p_ks = stats.kstest(df_server['response_latency_ms'], 'norm', 
                             args=(df_server['response_latency_ms'].mean(), df_server['response_latency_ms'].std()))

normality_res = pd.DataFrame({
    'Metode Uji': ['Shapiro-Wilk Test', 'Kolmogorov-Smirnov Test'],
    'Statistik Uji': [f"{stat_shapiro:.4f}", f"{stat_ks:.4f}"],
    'p-value': [f"{p_shapiro:.4f}", f"{p_ks:.4f}"],
    'Kesimpulan (α=0.05)': [
        'Distribusi Normal (H0 Diterima)' if p_shapiro > 0.05 else 'Tidak Berdistribusi Normal (H0 Ditolak)',
        'Distribusi Normal (H0 Diterima)' if p_ks > 0.05 else 'Tidak Berdistribusi Normal (H0 Ditolak)'
    ]
})

print("=== Hasil Uji Normalitas Latency Server ===")
display(normality_res)


## 📉 4. Visualisasi Diagnostik: Q-Q Plot dan Kurva Distribusi


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Q-Q Plot
stats.probplot(df_server['response_latency_ms'], dist="norm", plot=axes[0])
axes[0].set_title('Quantile-Quantile (Q-Q) Plot Latency Server', fontweight='bold')
axes[0].grid(True)

# Histogram & Fit Normal
sns.histplot(df_server['response_latency_ms'], kde=True, color='purple', ax=axes[1], stat="density")
axes[1].set_title('Kurva Kerapatan Densitas vs. Distribusi Teoretis', fontweight='bold')
axes[1].set_xlabel('Response Latency (ms)')

plt.tight_layout()
plt.show()


## ⚖️ 5. Uji Homogenitas Varians (Levene's Test) Antar-Cluster


In [ ]:
# Mengelompokkan latency berdasarkan 3 cluster server
group_alpha = df_server[df_server['cluster_group'] == 'Cluster_Alpha']['response_latency_ms']
group_beta = df_server[df_server['cluster_group'] == 'Cluster_Beta']['response_latency_ms']
group_gamma = df_server[df_server['cluster_group'] == 'Cluster_Gamma']['response_latency_ms']

stat_levene, p_levene = stats.levene(group_alpha, group_beta, group_gamma)

print(f"Statistik Levene: {stat_levene:.4f}, p-value: {p_levene:.4f}")
if p_levene > 0.05:
    print(">> Kesimpulan: Varians antar ketiga klaster server HOMOGEN (Asumsi Homoskedastisitas Terpenuhi).")
else:
    print(">> Kesimpulan: Varians antar klaster TIDAK HOMOGEN (Gunakan uji Welch).")


## 🔍 6. Uji Multikolinearitas (Variance Inflation Factor / VIF)


In [ ]:
# Memilih variabel prediktor numerik
features = ['cpu_usage_pct', 'ram_usage_gb', 'concurrent_requests']
X = df_server[features].dropna()
X_with_const = sm.add_constant(X)

vif_data = pd.DataFrame()
vif_data["Fitur Prediktor"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X_with_const.values, i+1) for i in range(len(features))]
vif_data["Status Multikolinearitas"] = vif_data["VIF"].apply(
    lambda x: "Aman (VIF < 5)" if x < 5 else ("Waspada (5 <= VIF < 10)" if x < 10 else "Tinggi (VIF >= 10)")
)

print("=== Hasil Analisis VIF ===")
display(vif_data)


## 📝 Kesimpulan Analisis

### Q&A
* **Mengapa uji asumsi wajib dilakukan sebelum pemodelan parametrik?** Karena estimasi koefisien parametrik (misal: OLS Regression, ANOVA, t-Test) mengasumsikan galat berdistribusi normal, varians homogen, dan tidak ada kolinearitas sempurna. Pelanggaran asumsi dapat menyebabkan kesimpulan palsu (*spurious conclusions*).

### Data Analysis Key Findings
* Uji normalitas Shapiro-Wilk pada variabel *response latency* menghasilkan $p$-value $> 0.05$, menandakan asumsi normalitas terpenuhi dengan baik.
* Nilai VIF untuk seluruh prediktor kinerja berada di bawah nilai ambang 5, mengindikasikan tidak adanya masalah multikolinearitas yang merusak model prediktif.

### Insights or Next Steps
* Dataset server ini valid dan memenuhi seluruh syarat formal untuk dianalisis lebih lanjut menggunakan regresi linier berganda atau ANOVA.
